In [1]:
from dotenv import load_dotenv
import os

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import wrap_model_call, wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import ToolMessage
from langgraph.types import Command

In [2]:
load_dotenv()

True

In [3]:
class EmailAgentState(AgentState):
    authenticated: bool 

In [4]:
@tool
def authenticate(username: str, password: str) -> str:
    """Authenticate the user."""

    if username == "dalia" and password == "1234":
        return "AUTHENTICATION_SUCCESS"

    return "AUTHENTICATION_FAILED"

In [5]:
@tool
def check_inbox() -> str:
    """Check the inbox for new emails."""

    return """
You have 2 new emails.

Email 1:
From: client@example.com
Subject: Project meeting

Email 2:
From: professor@example.com
Subject: Internship opportunity
"""

In [6]:
@tool
def read_email(email_id: str) -> str:
    """Read a specific email using its ID."""

    emails = {
        "1": """
Email ID: 1
From: client@example.com
Subject: Project meeting

Hello,

Can we schedule a meeting for Tuesday at 10 AM?

Best regards,
Client
""",

        "2": """
Email ID: 2
From: professor@example.com
Subject: Internship opportunity

Hello,

Please let me know if you are interested in discussing the internship.

Best regards,
Professor
"""
    }

    return emails.get(email_id, "Email not found.")

In [7]:
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""

    print("\n" + "=" * 50)
    print("📧 EMAIL SENT")
    print("=" * 50)
    print("To:", to)
    print("Subject:", subject)
    print("\nBody:")
    print(body)
    print("=" * 50)

    return f"EMAIL_SENT: Email successfully sent to {to}."

In [8]:
@wrap_tool_call
def authentication_state(request, handler):
    """Update authentication state after authentication."""

    result = handler(request)

    tool_name = request.tool_call["name"]

    if tool_name == "authenticate":

        if isinstance(result, ToolMessage):

            authentication_result = result.content

            if authentication_result == "AUTHENTICATION_SUCCESS":
                return Command(
                    update={
                        "messages": [result],
                        "authenticated": True,
                    }
                )

            return Command(
                update={
                    "messages": [result],
                    "authenticated": False,
                }
            )

    return result

In [9]:
@wrap_model_call
def dynamic_email_tools(request, handler):

    authenticated = request.state.get("authenticated", False)

    print("DEBUG - authenticated:", authenticated)

    if authenticated:

        print(
            "DEBUG - Available tools: "
            "check_inbox, read_email, send_email"
        )

        return handler(
            request.override(
                tools=[
                    check_inbox,
                    read_email,
                    send_email,
                ]
            )
        )

    print("DEBUG - Available tools: authenticate")

    return handler(
        request.override(
            tools=[
                authenticate,
            ]
        )
    )

In [10]:
model = init_chat_model(
    model="gemini-3.6-flash",
    model_provider="google_genai"
)

In [11]:
system_prompt = """
You are an email assistant.

You help the user manage their emails.

AUTHENTICATION:

- If the authenticate tool is available, the user is NOT authenticated.
- If check_inbox, read_email, or send_email are available, the user IS authenticated.
- Never ask the user to authenticate when authenticated tools are available.
- Never claim authentication succeeded unless the authenticate tool returned AUTHENTICATION_SUCCESS.

INBOX:

- When the user asks to check their inbox, ALWAYS call check_inbox.
- When the user asks to read an email, ALWAYS call read_email.
- Do not invent email contents.

REPLY WORKFLOW:

When the user asks you to reply to an email:

1. Read the email first if its contents are not already available.
2. Understand what the sender is asking.
3. Create a professional reply.
4. SHOW the proposed reply to the user.
5. Ask the user whether they want to send it.
6. DO NOT call send_email yet.

APPROVAL:

The user must explicitly approve the proposed email before it is sent.

Examples of approval:
- "Yes"
- "Send it"
- "Send"
- "I approve"
- "Go ahead"

If the user asks to modify the reply:
- Modify the proposed reply.
- Show the new version.
- Ask for approval again.

IMPORTANT:

NEVER call send_email unless the user has explicitly approved
the currently proposed email.

When send_email is called, use the recipient, subject,
and exact approved body.

Be concise and helpful.
"""

In [12]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

print("✅ Checkpointer created")

✅ Checkpointer created


In [13]:
agent = create_agent(
    model=model,
    tools=[
        authenticate,
        check_inbox,
        read_email,
        send_email,
    ],
    state_schema=EmailAgentState,
    middleware=[
        dynamic_email_tools,
        authentication_state,
    ],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
)

print("Agent created successfully.")

Agent created successfully.


In [14]:
config = {
    "configurable": {
        "thread_id": "dalia-email-test-001"
    }
}

print("✅ Thread configured")
print(config)

✅ Thread configured
{'configurable': {'thread_id': 'dalia-email-test-001'}}


In [16]:
print("\n" + "=" * 60)
print("TEST 1 — AUTHENTICATION")
print("=" * 60)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My username is dalia and my password is 1234."
            }
        ]
    },
    config=config
)

print("\n===== LAST MESSAGE =====")
print(response["messages"][-1].content)


TEST 1 — AUTHENTICATION
DEBUG - authenticated: False
DEBUG - Available tools: authenticate
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email

===== LAST MESSAGE =====
[{'type': 'text', 'text': 'You have successfully authenticated. How can I help you with your emails today?', 'extras': {'signature': 'EtADCs0DAWkUfRO8MNtuuUrVX4lxXjXGsRk8xxrv2Gz3nu4IF30KHcnmg7ftqxgy1K9dyNWprZ30LjPYCaR4adENQbKFRoyNWQqLPg+XuZfzAg0zQ9oajApNOMAYK5sEL1a9bkJlArOiiC4CsT/9J0hg2Yb1n7Y9VPQdJqx62r8W0pvl3RklFolfAppej98CcvloNhnsPCh0pJaR1wNABS2Cexi+E0rWY2I0v0aA9lzUZNXEUmUxaR/feiH57XIcq9RCdYldUb7xFcyKPHkgdL1vdhugK8+B8vP3zlucZq0DlinRsThMORHMk7vkJGA1X/DLB/jhdr6HhnA1Hr8QRUG0HOorI70SMjQXh22+DN92ouH0AuN2qi8WTwj34M3/8XrjyGaIvUJ9vjJlW1uMoM+/iW902BSoqvNNWOhfwwgxbdW/lRrRa4qGT8UaQM7l17kz+wvX2ynylBFLPuexubrpJYjU/tnVPSSOJKAM954DyWf6jsGJDACjU/JXzz5tpBxiNvmBe2u7Xe2q0ExILiodIYbBOtHBrm19sVy2KkTPGMmBZf/Xao68Ewjj3KLTpMqpkDfcuMVV3E7htXc7OTw/4VdBHfB3hidisbgRpzHe5mk='}}]


In [17]:
print("\n" + "=" * 60)
print("TEST 2 — CHECK SAVED STATE")
print("=" * 60)

state = agent.get_state(config)

print("Authenticated:", state.values.get("authenticated"))


TEST 2 — CHECK SAVED STATE
Authenticated: True


In [18]:
print("\n" + "=" * 60)
print("TEST 3 — CHECK INBOX")
print("=" * 60)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Check my inbox."
            }
        ]
    },
    config=config
)

print("\n===== LAST MESSAGE =====")
print(response["messages"][-1].content)


TEST 3 — CHECK INBOX
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email

===== LAST MESSAGE =====
[{'type': 'text', 'text': 'You have 2 new emails:\n\n1. **From:** client@example.com  \n   **Subject:** Project meeting  \n\n2. **From:** professor@example.com  \n   **Subject:** Internship opportunity  \n\nWould you like me to read either of these emails for you?', 'extras': {'signature': 'EtQCCtECAWkUfRMfEzq5I3Z4MEfUUEpgcN85wcK+RAddsJ1Hh6ZcQIl9tlrjKHJub7oUcW26qsT1DTXYuCfta6CNo2l2HjkFGvHHl1rrVcUpRFGjgoxUUJpSDbAD5PjQ+LKt6JuV5nXys+SCE6/QTM2rG2owHSCAJCO4iVKntuRrZC4GCZP3eAlvXKB8i+Gudyg9hEqAw504Reyz+ZHnc5R7IODQDGz3BJu98U7/VocM46u6rNtU83cBJG8gDonVubqvu8yIlOyLcppaO0N3xeoj901ingDQHw9yVEk1Q7IgxAbxcho3ABvogbX+fAuSEdEIHo9g0MHij9HEHR7/arZl9Pmhr+Fb8sFteiLImqIkw6/5jVPB/PlQCLunnJ5o93MhMYQErYnAW96E1YDkkPCmEe48pvRqTnuZKWhnywwfWPH+cZ/s/sYBGFVJ766AjvFyKE0yEA=='}}]


In [19]:
print("\n" + "=" * 60)
print("TEST 4 — READ EMAIL")
print("=" * 60)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Read email 1."
            }
        ]
    },
    config=config
)

print("\n===== LAST MESSAGE =====")
print(response["messages"][-1].content)


TEST 4 — READ EMAIL
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email

===== LAST MESSAGE =====
[{'type': 'text', 'text': 'Here is Email 1:\n\n**From:** client@example.com  \n**Subject:** Project meeting  \n\n**Body:**  \nHello,\n\nCan we schedule a meeting for Tuesday at 10 AM?\n\nBest regards,  \nClient', 'extras': {'signature': 'Et8CCtwCAWkUfRM6fr+AMFAQpqh/AWcpmp/htrLVN+gZNEePgueBXISYuDE+gXgtKaWtrMyfLamPRM+OK5VIwa22BHP4m3J00BPrTRolRpv9cl2ZM+DS6iTYMN/8RUi2E5ehpKohkcYzhH5CwPNR4KmoKq83G/HDkhmINCfbhvpiKrMsWSpH++53JbEnu/5OkqGGGqg0LiVYTmRqDIOFaSalW8D9Fbw1W9esOcujD/Cn2XT+OhWfejqnBOV2V5dVzqZvIzFh3u3MemhO5IRLmpVRezOm9HJ26b5rr+u2onISxTkA+bgJ4DHuDZh5Ll5fPf8qLvXo0K/tM69FT22FG2LljpaRzjLBVnmfBUPtb+X1g0W9vJrpEY+cvRoN5xqbmiUMyZmCzv7fXGy6qmkPDC9uX8pK14Qd9s9aQhxA2eL9zJVtOZ6p2LmtBzeVBx7FdInU/ibFiUWIe3QazGTIQJiG'}}]


In [20]:
print("\n" + "=" * 60)
print("TEST 5 — CREATE EMAIL PROPOSAL")
print("=" * 60)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Create a professional reply to email 1.

Do NOT send the email yet.
Show me the complete proposed email and ask me for approval.
"""
            }
        ]
    },
    config=config
)

print("\n===== PROPOSAL =====")
print(response["messages"][-1].content)


TEST 5 — CREATE EMAIL PROPOSAL
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email

===== PROPOSAL =====
[{'type': 'text', 'text': 'Here is the proposed reply:\n\n**To:** client@example.com  \n**Subject:** Re: Project meeting  \n\n**Proposed Body:**  \nDear Client,\n\nThank you for reaching out. Tuesday at 10:00 AM works well for me. I look forward to our meeting.\n\nBest regards,  \nDalia\n\n---\n\nWould you like me to send this email, or would you like to make any modifications first?', 'extras': {'signature': 'EvcGCvQGAWkUfROlnuJgaBpIDO8lWBZ8FT4GbcMWYWZVmbDgj1KWQidLAjuc7MABUbEcjGzLEHbmlm/UVVJAaanjT5J9fo2AN6w5zm07mEuFM9vK5MdzuxIuhQoPoi+Xm1bd8mPzTBAHiqbBNi9DyiPMywqXcSbKFkO8rjFlaS6gN4ZB7wm09GFT8Ct4rYRnwPcZp/QyY6zcvJOAmDSWc8gspnBOGgZjjMO0E8z2ZSY6fHiq5tMoKzc8Y379WJJeHbTd/cDMqw85Q6n2txv9FJD/9Dg27nQr6v4AD1nupYyrEUrZ9SYrhJs0BJQRal23ygqObp3+LvqoZIS4K1VCGB3s1EBjqgHxnxDeba7xYECmgs22/fqW79b7blWQIwzsZG6ZwrTTGnHc7nG5WlLGefOaJjfKnjlsdfIunTd8C2PZ3dYj2MpicOOqOLD

In [21]:
print("\n" + "=" * 60)
print("TEST 6 — STATE BEFORE APPROVAL")
print("=" * 60)

state = agent.get_state(config)

print("Authenticated:", state.values.get("authenticated"))

print("\nMessages stored:", len(state.values.get("messages", [])))


TEST 6 — STATE BEFORE APPROVAL
Authenticated: True

Messages stored: 14


In [22]:
print("\n" + "=" * 60)
print("TEST 7 — USER APPROVAL")
print("=" * 60)

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Yes, send it."
            }
        ]
    },
    config=config
)

print("\n===== FINAL RESPONSE =====")
print(response["messages"][-1].content)


TEST 7 — USER APPROVAL
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email

📧 EMAIL SENT
To: client@example.com
Subject: Re: Project meeting

Body:
Dear Client,

Thank you for reaching out. Tuesday at 10:00 AM works well for me. I look forward to our meeting.

Best regards,
Dalia
DEBUG - authenticated: True
DEBUG - Available tools: check_inbox, read_email, send_email

===== FINAL RESPONSE =====
[{'type': 'text', 'text': 'Your email has been successfully sent to client@example.com! Let me know if you need help with anything else.', 'extras': {'signature': 'EqYBCqMBAWkUfRN5xuGdHFj9IR7iD68IPrgHCGFkAdZvo3gVem1duHCvfQ1bCnpqzro9e6OfSvTqb4ilWtg7vuMRXxUvAGuZNcOWLsyLoBWh3NKTjVAk/8GWaQCgZF+SSADwX5SDsUgy6i0Dn+Uo0W11wYhoavVI8rA0/U1sYlIy4DYvb8hZS9aToWKD0kRoWNeD4zFPXwn7YTUw7NPqX4u3pNpDyy0bRQ=='}}]


In [23]:
print("=" * 60)
print("FINAL STATE")
print("=" * 60)

state = agent.get_state(config)

print("Authenticated:", state.values.get("authenticated"))

print("\nMessages:")

for i, msg in enumerate(state.values.get("messages", [])):
    print(f"\n--- {i} | {type(msg).__name__} ---")
    print(msg.content)

    if hasattr(msg, "tool_calls") and msg.tool_calls:
        print("TOOL CALLS:")
        print(msg.tool_calls)

FINAL STATE
Authenticated: True

Messages:

--- 0 | HumanMessage ---
My username is dalia and my password is 1234.

--- 1 | AIMessage ---
[]
TOOL CALLS:
[{'name': 'authenticate', 'args': {'username': 'dalia', 'password': '1234'}, 'id': 'call_227220', 'type': 'tool_call'}]

--- 2 | ToolMessage ---
AUTHENTICATION_SUCCESS

--- 3 | AIMessage ---
[{'type': 'text', 'text': 'You have successfully authenticated. How can I help you with your emails today?', 'extras': {'signature': 'EtADCs0DAWkUfRO8MNtuuUrVX4lxXjXGsRk8xxrv2Gz3nu4IF30KHcnmg7ftqxgy1K9dyNWprZ30LjPYCaR4adENQbKFRoyNWQqLPg+XuZfzAg0zQ9oajApNOMAYK5sEL1a9bkJlArOiiC4CsT/9J0hg2Yb1n7Y9VPQdJqx62r8W0pvl3RklFolfAppej98CcvloNhnsPCh0pJaR1wNABS2Cexi+E0rWY2I0v0aA9lzUZNXEUmUxaR/feiH57XIcq9RCdYldUb7xFcyKPHkgdL1vdhugK8+B8vP3zlucZq0DlinRsThMORHMk7vkJGA1X/DLB/jhdr6HhnA1Hr8QRUG0HOorI70SMjQXh22+DN92ouH0AuN2qi8WTwj34M3/8XrjyGaIvUJ9vjJlW1uMoM+/iW902BSoqvNNWOhfwwgxbdW/lRrRa4qGT8UaQM7l17kz+wvX2ynylBFLPuexubrpJYjU/tnVPSSOJKAM954DyWf6jsGJDACjU/JXzz5tpBxiNvmBe2